In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 1 — Install dependencies                               ║
# ╚══════════════════════════════════════════════════════════════╝
# transformers==5.2.0 requires huggingface_hub>=1.0.0, but 1.x adds strict
# validate_repo_id that rejects local /kaggle/input/... paths (>1 slash).
# 4.46.3 handles local paths via os.path.isdir before touching hf_hub,
# so it works correctly with huggingface_hub 1.x that gradio installs.
!pip install transformers==4.46.3 peft==0.18.1 sentencepiece joblib gradio --quiet

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 2 — Configuration                                      ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Model paths ───────────────────────────────────────────────────────────────
CLASSIFIER_DIR  = "/kaggle/input/datasets/austraa/bangla-bert-dialect"
BASE_MODEL_PATH = "/kaggle/input/datasets/farhanaadri/nllb-dialect-model"

# Adapter dataset paths — one per dialect
# Each path should point to the folder containing adapter_config.json
ADAPTER_PATHS = {
    "chittagong" : "/kaggle/input/datasets/mahbubasayedsinthia/chittagong-adapter-hp",
    "sylhet"     : "/kaggle/input/datasets/mahbubasayedsinthia/sylhet-adapter-hp",
    "noakhali"   : "/kaggle/input/datasets/mahbubasayedsinthia/noakhali-adapter-hp",
    "barisal"    : "/kaggle/input/datasets/mahbubasayedsinthia/barisal-adapter-hp",
    "mymensingh" : "/kaggle/input/datasets/abonykamal/mymensingh-adapter-hp",
}

# ── Dialect metadata ──────────────────────────────────────────────────────────
# Maps BanglaBERT output label → (adapter_name, NLLB src_lang_tag)
# Standard Bangla has no adapter → adapter_name is None
DIALECT_CONFIG = {
    "Chittagong"     : {"adapter": "chittagong",  "src_lang": "bng_chittagong"},
    "Sylhet"         : {"adapter": "sylhet",       "src_lang": "bng_sylhet"},
    "Noakhali"       : {"adapter": "noakhali",     "src_lang": "bng_noakhali"},
    "Barisal"        : {"adapter": "barisal",      "src_lang": "bng_barisal"},
    "Mymensingh"     : {"adapter": "mymensingh",   "src_lang": "bng_mymensingh"},
    "Standard Bangla": {"adapter": None,           "src_lang": "ben_Beng"},
}

# ── Generation hyperparameters ────────────────────────────────────────────────
# Using the winning configs per dialect from your HP search
GEN_CONFIG = {
    "chittagong" : {"num_beams": 3,  "length_penalty": 0.8},
    "sylhet"     : {"num_beams": 8,  "length_penalty": 1.0},
    "noakhali"   : {"num_beams": 3,  "length_penalty": 0.8},
    "barisal"    : {"num_beams": 3,  "length_penalty": 0.8},
    "mymensingh" : {"num_beams": 3,  "length_penalty": 0.8},
    "standard"   : {"num_beams": 4,  "length_penalty": 1.0},
}

MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 128
TGT_LANG       = "eng_Latn"

# ── Confidence threshold ──────────────────────────────────────────────────────
# If classifier confidence is below this, log a warning but still proceed
CONFIDENCE_THRESHOLD = 0.60

print("Configuration loaded.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 3 — Imports and normalization utilities                ║
# ╚══════════════════════════════════════════════════════════════╝
import os, re, unicodedata, warnings
import numpy as np
import torch
import joblib
from contextlib import contextmanager

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
)
from peft import PeftModel

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'N/A'}")


# ── Text normalization ────────────────────────────────────────────────────────
# Two functions: one for the classifier (aggressive, matches training),
# one for NLLB (dialect-safe, whitespace/punctuation only).

def normalize_for_classifier(text: str) -> str:
    """
    Matches the normalization used during BanglaBERT training.
    Aggressive: NFKC, zero-width removal, lowercase.
    """
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = text.replace("৷", "।")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


_PUNCT_MAP = str.maketrans({
    "\u201c": '"', "\u201d": '"',
    "\u2018": "'", "\u2019": "'",
    "\u2013": "-", "\u2014": "-",
    "\u2026": "...",
    "\u00a0": " ",
    "\u200b": "", "\u200c": "", "\u200d": "", "\ufeff": "",
})

def normalize_for_translation(text: str) -> str:
    """
    Dialect-safe normalization for NLLB input.
    Whitespace and punctuation only — preserves dialectal features.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PUNCT_MAP)
    text = re.sub(r"[\u200b-\u200f\ufeff]", "", text)
    text = re.sub(r" +", " ", text).strip()
    return text


print("Normalization functions ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 4 — Load BanglaBERT dialect classifier                 ║
# ╚══════════════════════════════════════════════════════════════╝

print(f"Loading classifier from: {CLASSIFIER_DIR}")

cls_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_DIR)
cls_model     = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_DIR)
label_encoder = joblib.load(os.path.join(CLASSIFIER_DIR, "label_encoder.joblib"))

cls_model.to(device)
cls_model.eval()

print(f"  Classes : {list(label_encoder.classes_)}")
print(f"  Params  : {sum(p.numel() for p in cls_model.parameters()):,}")
print("Classifier ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 5 — Load NLLB base model + all 5 LoRA adapters        ║
# ╚══════════════════════════════════════════════════════════════╝
# Strategy: load all adapters into the model at startup using PEFT's
# multi-adapter API. Hot-swapping with set_adapter() is then ~instant.
# Standard Bangla uses disable_adapter() to run the frozen base model.

print(f"Loading NLLB base model from: {BASE_MODEL_PATH}")
nllb_tokenizer = NllbTokenizer.from_pretrained(BASE_MODEL_PATH, use_fast=False)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ignore_mismatched_sizes=True,
)
print(f"  Base model params: {sum(p.numel() for p in base_model.parameters()):,}")

# Load the first adapter to initialise the PeftModel wrapper
first_dialect = list(ADAPTER_PATHS.keys())[0]
print(f"\nLoading adapter: {first_dialect}")
nllb_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATHS[first_dialect],
    adapter_name=first_dialect,
)

# Load the remaining adapters into the same PeftModel
for dialect, path in list(ADAPTER_PATHS.items())[1:]:
    print(f"Loading adapter: {dialect}")
    nllb_model.load_adapter(path, adapter_name=dialect)

nllb_model.to(device)
nllb_model.eval()

# Sanity check: list loaded adapters
loaded = list(nllb_model.peft_config.keys())
print(f"\nAll adapters loaded: {loaded}")

# Verify dialect tags exist in tokenizer
for dialect, cfg in DIALECT_CONFIG.items():
    tag_id = nllb_tokenizer.convert_tokens_to_ids(cfg["src_lang"])
    status = "✅" if tag_id != nllb_tokenizer.unk_token_id else "❌ MISSING"
    print(f"  {cfg['src_lang']:20s} → token_id {tag_id}  {status}")

print("\nNLLB model + all adapters ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 6 — Inference pipeline class                           ║
# ╚══════════════════════════════════════════════════════════════╝

class BengaliDialectTranslator:
    """
    End-to-end pipeline: Bengali dialect text → English translation.

    Steps:
      1. Classify dialect with BanglaBERT
      2. Select the correct LoRA adapter (or disable for Standard Bangla)
      3. Translate with NLLB + selected adapter
    """

    def __init__(
        self,
        cls_model,
        cls_tokenizer,
        label_encoder,
        nllb_model,
        nllb_tokenizer,
        dialect_config,
        gen_config,
        device,
        max_source_len=128,
        max_target_len=128,
        confidence_threshold=0.60,
    ):
        self.cls_model    = cls_model
        self.cls_tok      = cls_tokenizer
        self.le           = label_encoder
        self.nllb         = nllb_model
        self.nllb_tok     = nllb_tokenizer
        self.dialect_cfg  = dialect_config
        self.gen_cfg      = gen_config
        self.device       = device
        self.max_src      = max_source_len
        self.max_tgt      = max_target_len
        self.conf_thresh  = confidence_threshold
        self.tgt_lang_id  = nllb_tokenizer.convert_tokens_to_ids(TGT_LANG)

    # ── Step 1: classify ──────────────────────────────────────────────────────
    def classify(self, text: str) -> dict:
        """Returns dialect label, confidence, and full probability distribution."""
        clean = normalize_for_classifier(text)
        inputs = self.cls_tok(
            clean,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128,
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = self.cls_model(**inputs).logits
            probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

        pred_id    = int(np.argmax(probs))
        pred_label = self.le.inverse_transform([pred_id])[0]
        confidence = float(probs[pred_id])

        all_probs = {
            self.le.inverse_transform([i])[0]: float(p)
            for i, p in enumerate(probs)
        }

        if confidence < self.conf_thresh:
            print(
                f"  ⚠️  Low confidence ({confidence:.2%}) for '{pred_label}'. "
                f"Result may be unreliable."
            )

        return {
            "dialect"    : pred_label,
            "confidence" : confidence,
            "all_probs"  : all_probs,
        }

    # ── Step 2: select adapter ────────────────────────────────────────────────
    @contextmanager
    def _adapter_context(self, dialect_label: str):
        """
        Context manager that activates the right adapter for the given dialect,
        then restores state on exit. Thread-unsafe but fine for single-process
        Kaggle inference.
        """
        cfg          = self.dialect_cfg[dialect_label]
        adapter_name = cfg["adapter"]

        if adapter_name is None:
            # Standard Bangla: run the frozen base model with no adapter
            with self.nllb.disable_adapter():
                yield cfg
        else:
            self.nllb.set_adapter(adapter_name)
            yield cfg

    # ── Step 3: translate ─────────────────────────────────────────────────────
    def translate(self, text: str, dialect_label: str) -> str:
        """
        Translates a single string using the adapter for the specified dialect.
        You can call this directly if you already know the dialect.
        """
        cfg        = self.dialect_cfg[dialect_label]
        gen_params = self.gen_cfg.get(
            cfg["adapter"] or "standard",
            {"num_beams": 4, "length_penalty": 1.0}
        )
        clean = normalize_for_translation(text)

        with self._adapter_context(dialect_label):
            self.nllb_tok.src_lang = cfg["src_lang"]
            inputs = self.nllb_tok(
                clean,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=self.max_src,
            ).to(self.device)

            with torch.no_grad():
                generated = self.nllb.generate(
                    **inputs,
                    forced_bos_token_id = self.tgt_lang_id,
                    max_length          = self.max_tgt,
                    num_beams           = gen_params["num_beams"],
                    length_penalty      = gen_params["length_penalty"],
                    early_stopping      = True,
                )

        translation = self.nllb_tok.decode(generated[0], skip_special_tokens=True)
        return translation

    # ── Full pipeline: classify + translate ───────────────────────────────────
    def __call__(self, text: str, return_details: bool = False):
        """
        Full pipeline. Returns translation string by default.
        Set return_details=True to get the full result dict.
        """
        cls_result  = self.classify(text)
        dialect     = cls_result["dialect"]
        translation = self.translate(text, dialect)

        result = {
            "input"       : text,
            "dialect"     : dialect,
            "confidence"  : round(cls_result["confidence"], 4),
            "translation" : translation,
            "all_probs"   : {k: round(v, 4) for k, v in cls_result["all_probs"].items()},
        }

        if return_details:
            return result
        return translation

    # ── Batch inference ───────────────────────────────────────────────────────
    def translate_batch(self, texts: list[str], batch_size: int = 16) -> list[dict]:
        """
        Classifies and translates a list of texts.
        Groups by dialect to minimise adapter switches (1 switch per dialect group).
        """
        # Step 1: classify all texts
        print(f"Classifying {len(texts)} texts...")
        classified = []
        for text in texts:
            cls = self.classify(text)
            classified.append((text, cls["dialect"], cls["confidence"]))

        # Step 2: group indices by dialect to batch same-dialect inputs together
        from collections import defaultdict
        dialect_groups = defaultdict(list)
        for i, (text, dialect, conf) in enumerate(classified):
            dialect_groups[dialect].append(i)

        print(f"Dialect distribution: { {d: len(idxs) for d, idxs in dialect_groups.items()} }")

        # Step 3: translate each group in batches
        translations = [""] * len(texts)

        for dialect, indices in dialect_groups.items():
            cfg        = self.dialect_cfg[dialect]
            gen_params = self.gen_cfg.get(
                cfg["adapter"] or "standard",
                {"num_beams": 4, "length_penalty": 1.0}
            )
            print(f"  Translating {len(indices)} {dialect} samples "
                  f"(beams={gen_params['num_beams']})...")

            with self._adapter_context(dialect):
                self.nllb_tok.src_lang = cfg["src_lang"]

                for start in range(0, len(indices), batch_size):
                    batch_indices = indices[start : start + batch_size]
                    batch_texts   = [
                        normalize_for_translation(classified[i][0])
                        for i in batch_indices
                    ]

                    inputs = self.nllb_tok(
                        batch_texts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=self.max_src,
                    ).to(self.device)

                    with torch.no_grad():
                        generated = self.nllb.generate(
                            **inputs,
                            forced_bos_token_id = self.tgt_lang_id,
                            max_length          = self.max_tgt,
                            num_beams           = gen_params["num_beams"],
                            length_penalty      = gen_params["length_penalty"],
                            early_stopping      = True,
                        )

                    decoded = self.nllb_tok.batch_decode(
                        generated, skip_special_tokens=True
                    )
                    for j, idx in enumerate(batch_indices):
                        translations[idx] = decoded[j]

        # Assemble results
        return [
            {
                "input"      : classified[i][0],
                "dialect"    : classified[i][1],
                "confidence" : round(classified[i][2], 4),
                "translation": translations[i],
            }
            for i in range(len(texts))
        ]


print("Pipeline class defined.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 7 — Instantiate the pipeline                           ║
# ╚══════════════════════════════════════════════════════════════╝

pipeline = BengaliDialectTranslator(
    cls_model             = cls_model,
    cls_tokenizer         = cls_tokenizer,
    label_encoder         = label_encoder,
    nllb_model            = nllb_model,
    nllb_tokenizer        = nllb_tokenizer,
    dialect_config        = DIALECT_CONFIG,
    gen_config            = GEN_CONFIG,
    device                = device,
    max_source_len        = MAX_SOURCE_LEN,
    max_target_len        = MAX_TARGET_LEN,
    confidence_threshold  = CONFIDENCE_THRESHOLD,
)

print("Pipeline ready ✅")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 8 — Gradio demo app                                    ║
# ╚══════════════════════════════════════════════════════════════╝
import gradio as gr

# One authentic sentence per dialect, pulled from the test CSVs
EXAMPLES = [
    ["মোর ঘোরতে বেমালা ভালো লাগে"],                              # Barisal
    ["আই একদিন স্কুলর মাডত মাথা গুরাইয়েরে ফরি গেইলাম"],         # Chittagong
    ["বিয়ার লাইজ্ঞা আম্মা আমার লাইজ্ঞা ছেড়া খুজতাসে"],           # Mymensingh
    ["তুই কি আরে এই কাম আন করি দিতা হাইরবা নি?"],               # Noakhali
    ["তোমার মত খারাফ পুয়া মুই আর একটাও দেখছি না"],              # Sylhet
]


def translate_fn(text: str):
    if not text or not text.strip():
        return "", ""
    result = pipeline(text, return_details=True)
    return (
        result["dialect"],
        result["translation"],
    )


with gr.Blocks(title="Bengali Dialect → English Translator") as demo:
    gr.Markdown(
        """
        # Bengali Dialect → English Translator
        Paste Bengali dialect text, or click an example sentence below, then press **Translate**.  
        Detects and translates: **Barisal · Chittagong · Mymensingh · Noakhali · Sylhet · Standard Bangla**
        """
    )

    with gr.Row():
        txt_input = gr.Textbox(
            label="Bengali Dialect Text",
            placeholder="বাংলা লিখুন…",
            lines=3,
        )

    btn = gr.Button("Translate", variant="primary")

    with gr.Row():
        out_dialect = gr.Textbox(label="Detected Dialect", interactive=False)

    out_translation = gr.Textbox(
        label="English Translation",
        interactive=False,
        lines=3,
    )

    gr.Examples(
        examples=EXAMPLES,
        inputs=txt_input,
        label="Example sentences — one per dialect (click to load)",
    )

    btn.click(
        fn=translate_fn,
        inputs=txt_input,
        outputs=[out_dialect, out_translation],
        concurrency_limit=1,   # serialise requests — set_adapter() is not thread-safe
    )

demo.launch(share=True)